In [1]:
# Use with `Bash (micromamba)`
# Sync data files for streamlit app
SRC='euler.ethz.ch:/cluster/home/jjaenes/af2genomics/results/human-mutfunc-data/'
DEST='./data/'
ARGS='--progress '
#ARGS+='--delete '
ARGS+='--dry-run '
rsync -azv $ARGS "$SRC" "$DEST" \
    --include='/missense.parquet' \
    --include='/monomers-db' \
    --include='/monomers-db.dbtype' \
    --include='/monomers-db.index' \
    --include='/monomers-db.lookup' \
    --include='/multimers-db' \
    --include='/multimers-db.dbtype' \
    --include='/multimers-db.index' \
    --include='/multimers-db.lookup' \
    --include='/pockets.parquet' \
    --exclude='*'
du -hs data/

receiving incremental file list

sent 260 bytes  received 306 bytes  161.71 bytes/sec
total size is 6,766,068,189  speedup is 11,954,184.08 (DRY RUN)
6.3G	data/


In [ ]:
# Generate tag/version
SHORT_HASH="$(date +%y.%m)-$(git rev-parse --short HEAD)"
if ! git diff --quiet && git diff --cached --quiet; then
    SHORT_HASH+="-dirty"
fi
echo $SHORT_HASH

26.06-7589aac-dirty


In [ ]:
# Build container image with /data included
PROGRESS='--progress plain'
IMAGE="--tag mutfunc:$SHORT_HASH"
PLATFORM='--platform=linux/amd64'
#PLATFORM='--platform=linux/arm64'
#PLATFORM='--platform=linux/arm64,linux/amd64'
docker buildx build $PROGRESS $IMAGE $PLATFORM .

#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 1.62kB done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.13-slim
#2 ...

#3 [auth] library/python:pull token for registry-1.docker.io
#3 DONE 0.0s

#4 [internal] load metadata for ghcr.io/astral-sh/uv:latest
#4 ...

#2 [internal] load metadata for docker.io/library/python:3.13-slim
#2 DONE 0.7s

#4 [internal] load metadata for ghcr.io/astral-sh/uv:latest
#4 DONE 1.1s

#5 [internal] load .dockerignore
#5 transferring context: 2B done
#5 DONE 0.0s

#6 [stage-0  1/12] FROM docker.io/library/python:3.13-slim@sha256:c33f0bc4364a6881bed1ec0cc2665e6c53c87a43e774aaeab88e6f17af105e4f
#6 resolve docker.io/library/python:3.13-slim@sha256:c33f0bc4364a6881bed1ec0cc2665e6c53c87a43e774aaeab88e6f17af105e4f 0.0s done
#6 DONE 0.0s

#7 FROM ghcr.io/astral-sh/uv:latest@sha256:d0a0a753ab981624b49c97abc98821c1c09f4ca69d1ef5cee69c501be3d884

In [ ]:
# Quick test - emulation so will be slow..
docker run -p 8501:8501 mutfunc:$SHORT_HASH

: 1

In [38]:
# Save image as tar archive that can be copied over to the host VM
docker save mutfunc:$SHORT_HASH -o mutfunc-$SHORT_HASH.tar

In [40]:
# Copying image tar archive to host VM
SRC='./'
DEST='sysbc-lx-s04.ethz.ch:/images'
ARGS='--progress '
#ARGS+='--delete '
#ARGS+='--dry-run '
rsync -azv $ARGS "$SRC" "$DEST" \
    --include="mutfunc-$SHORT_HASH.tar" \
    --exclude='*'

sending incremental file list
./
mutfunc-26.06-7589aac-dirty.tar
  7,071,017,472 100%  110.70MB/s    0:01:00 (xfr#1, to-chk=0/2)

sent 7,068,540,950 bytes  received 38 bytes  113,096,655.81 bytes/sec
total size is 7,071,017,472  speedup is 1.00


In [ ]:
# On host VM, load image from tar archive - https://docs.docker.com/reference/cli/docker/image/load/
# $ ssh sysbc-lx-s04.ethz.ch
echo $ sudo docker load --input /images/mutfunc-$SHORT_HASH.tar

$ sudo docker load --input /images/mutfunc-26.06-7589aac-dirty.tar


In [44]:
# On host VM, stop old image and start new one
# $ sudo docker ps
# $ sudo docker stop <container_name_or_id>
echo $ sudo docker run -d -p 8501:8501 mutfunc:$SHORT_HASH

$ sudo docker run -d -p 8501:8501 mutfunc:26.06-7589aac-dirty


In [47]:
# Clean up local images if needd
# $ docker system prune --all --volumes
docker images

REPOSITORY   TAG       IMAGE ID   CREATED   SIZE
